# Module 1: What you are building, and how to see inside it

You run the prototype agent once and look inside it: the raw request and response for each model call, and the run's trace in MLflow. Every later module uses these two kinds of evidence.

In [ ]:
import labnb as lab
%cd ../agent

## Start from module 1

This cell puts the agent back where module 1 starts. It copies any edits you made to
`agent/.reset-backup/` first, so nothing you wrote is lost.

In [ ]:
!python reset.py 1

## 1. Meet the agent

The agent answers field service questions for Kestrel Industrial, a fictional maker of industrial
cooling equipment. The lab folder in your workbench, `agent-antipatterns-lab`, has the two parts you'll
use:

```text
agent-antipatterns-lab/
├── notebooks/         the modules, one notebook each
└── agent/             the agent you'll fix
    ├── agent.py       the loop
    ├── tools.py       the tools the model is offered
    ├── config.toml    which model endpoint it calls
    ├── runs/          the payloads each run saves (created by the first run)
    └── ...            other files you'll meet in later modules
```

The first code cell above changed the working directory to `agent/`, so commands such as
`python agent.py` run there. Click a file name to open it: [`agent.py`](../agent/agent.py), [`tools.py`](../agent/tools.py),
[`config.toml`](../agent/config.toml).

The model the agent calls runs on the cluster, not in your workbench. It's Qwen3-8B, served by vLLM,
an open-source inference server, through OpenShift AI's model serving. The lab has two endpoints with
the same model, and `config.toml` points the agent at one of them.

Each run leaves two kinds of evidence: the raw payloads under `agent/runs/`, and a trace in MLflow.

```mermaid
flowchart LR
  Q["Your question"] --> A["agent.py: the loop"]
  A <-- "request and response" --> M["Model: qwen3-8b-a on vLLM"]
  A <-- "tool calls and results" --> T["tools.py: find_part, check_stock"]
  A --> ANS["The answer it prints"]
  A -. "payloads" .-> R["agent/runs/"]
  A -. "one trace per run" .-> F["MLflow"]
```

`config.toml` sets the endpoint the lab starts on. Leave it as it is for now:

In [ ]:
# Which model endpoint config.toml points the agent at.
print(lab.endpoint())

## 2. Run it once

A field engineer is helping a customer whose K-400
shows a coolant pressure fault, and needs the part number for a replacement coolant pump. The lab keeps
its test questions in [`questions.py`](../agent/questions.py) and calls this one `m1`, so the command
below asks it with `--q m1`.

The question needs a part number. Only `find_part` has it, because Kestrel's parts aren't in any
model's training data.

In [ ]:
# Ask the agent the lab's question 'm1':
#   "A customer's K-400 is showing a coolant pressure fault. What's the replacement part number
#   for the coolant pump?"
!python agent.py --q m1
# Show a table of this notebook's runs, one row each:
#   endpoint: which model endpoint answered
#   finish_reason (call 1): how the first model call ended
#   tool_calls (call 1): the tools the first model call asked for
#   turns: how many times the agent called the model
#   answer: what the agent printed
lab.evidence(columns=("endpoint", "finish_reason (call 1)", "tool_calls (call 1)", "turns", "answer"))

**Your read:** look at the answer. Did the agent look anything up? Did anything report a problem?

## 3. Read the payload

Each time the agent calls the model, it saves two files under `agent/runs/`: the request it sent and
the response that came back. Together they're the payload. The next cell prints a short digest of
each model call. Here's what its lines mean:

- `request`: how many messages the agent sent, and which tools it declared. Declared tools are the
  ones the model may ask the agent to run.
- `finish_reason`: why the model stopped writing. `stop` means it finished its answer. `tool_calls`
  means it stopped to ask the agent to run a tool.
- `tool_calls`: the tools the model asked for, as structured data the agent can act on. `null` or
  `[]` means it asked for none.
- `content`: the text the model wrote.

For comparison, here's a model call that does ask for a tool, from a run in module 2 (the id is shortened):

```text
response: finish_reason = tool_calls
          tool_calls    = [{"id": "chatcmpl-tool-…", "type": "function", "function": {"name": "find_part", "arguments": "{\"query\": \"coolant pump\", \"product\": \"K-400\"}"}}]
          content       = null
```

In [ ]:
!python payloads.py

In [ ]:
lab.content()

Now compare your run with that. The request declared the tools. In the response, `tool_calls` is
empty and `finish_reason` is `stop`, so as far as the agent can tell, the model asked for nothing.
But `content` holds the call the model meant to make, written out as text. Module 2 works out why.

## 4. Your first trace

MLflow is an open-source tool for tracking machine learning work, and it runs as part of OpenShift AI.
This lab uses it to record what the agent does. Every run of the agent sends MLflow a **trace**, a
record of each step that you can open in your browser. Your traces go to your own workspace, named
after your project. If MLflow asks you to log in, use the username and password from your lab's provisioning details.

A trace is a tree of **spans**, one per step:

- `agent_run` is the whole run. Its outputs say how the run ended and how many tools it called.
- `model_call_1` is one request to the model. Its **Inputs** hold the messages and the tools offered;
  its **Outputs** hold `finish_reason`, `tool_calls` and `content`.
- A tool the agent actually runs gets its own span underneath. This run should have none.

The first link below opens this run's trace. Click each span to see its inputs and outputs, and
compare what `model_call_1` shows with the payload above. The second link opens the list of all your
traces, one row per run.

**This is the module's success signal:** you can find the same `content` text in the trace's
`model_call_1` outputs as in the payload.

In [ ]:
# A link to the newest run's trace in MLflow.
lab.trace_link()
lab.links(("Every trace in this experiment", lab.experiment_url()))

## 5. The rule for the rest of the lab

From here on, every module ends by checking the evidence, not the answer: the payload in this module
and the next, then the trace from module 3 on. In the trace, look at the number of turns, the order
of the tool calls and the error categories.